In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE  = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
PROC  = BASE / "data/processed"
TABS  = BASE / "outputs/tables"

print("Imports OK")
checks = {}

In [ ]:
print("\n── V1: Gráfico 2 — distribuição de jornada ─────────────")

df_pnadc = pd.read_csv(PROC / "pnadc_2023_anual.csv", dtype=str)
df_pnadc["VD4031"] = pd.to_numeric(df_pnadc["VD4031"], errors="coerce")
ocupados = df_pnadc[
    (df_pnadc["VD4002"].str.strip() == "1") &
    (df_pnadc["VD4031"].notna()) &
    (df_pnadc["VD4031"] > 0)
].copy()

bins   = [0, 20, 30, 39, 40, 44, 48, 60, 200]
labels = ["≤20h", "21–30h", "31–39h", "40h", "41–44h", "45–48h", "49–60h", "60h+"]
ocupados["faixa"] = pd.cut(ocupados["VD4031"], bins=bins, labels=labels, right=True)

dist = ocupados["faixa"].value_counts().sort_index()
dist_pct = (dist / dist.sum() * 100).round(1)

print("  Distribuição de jornada (valores para o gráfico):")
for faixa, pct in dist_pct.items():
    print(f"    {faixa}: {pct}%")

pct_40h_exatas = dist_pct.get("40h", 0)
pct_41_44h     = dist_pct.get("41–44h", 0)
pct_acima_44h  = dist_pct.get("45–48h", 0) + dist_pct.get("49–60h", 0) + dist_pct.get("60h+", 0)
pct_abaixo_40  = dist_pct.get("≤20h", 0) + dist_pct.get("21–30h", 0) + dist_pct.get("31–39h", 0)

print(f"\n  Valores-chave derivados (sem hardcode):")
print(f"    Abaixo de 40h:    {pct_abaixo_40:.1f}%")
print(f"    Exatamente 40h:   {pct_40h_exatas:.1f}%")
print(f"    Faixa 41–44h:     {pct_41_44h:.1f}%  ← diretamente afetados")
print(f"    Acima de 44h:     {pct_acima_44h:.1f}%")
print(f"    Total:            {pct_abaixo_40+pct_40h_exatas+pct_41_44h+pct_acima_44h:.1f}%")

checks["V1_distribuicao_jornada"] = abs(
    pct_abaixo_40 + pct_40h_exatas + pct_41_44h + pct_acima_44h - 100
) < 0.5

In [ ]:
print("\n── V2: Gráfico 12 — heatmap ─────────────────────────────")

df_mat = pd.read_csv(TABS / "matriz_hipoteses_completa.csv")

bl_0 = df_mat[
    (df_mat["faixa"] == "41–44h") & (df_mat["ganho_p"] == "0%")
]["delta_pib_incl"].values[0]

bl_max = df_mat[
    df_mat["categoria"] == "baseline_consistente"
]["delta_pib_incl"].max()

print(f"  Baseline 41–44h, ganho 0%:  {bl_0:.4f}%")
print(f"  Baseline máximo (+3%):      {bl_max:.4f}%")
print(f"  Faixa baseline:             [{df_mat[df_mat['categoria']=='baseline_consistente']['delta_pib_incl'].min():.4f}%, {bl_max:.4f}%]")
print(f"  CNI (-0,700%) distância:    {abs(bl_0 - (-0.700)):.4f} pp")

categorias_presentes = df_mat["categoria"].unique().tolist()
print(f"\n  Categorias na matriz: {categorias_presentes}")

checks["V2_heatmap_consistente"] = bl_0 is not None and len(df_mat) == 16

In [ ]:
print("\n── V3: Gráfico 13 — normalização setorial ───────────────")

df_norm = pd.read_csv(TABS / "normalizacao_setorial.csv")

incl = df_norm[df_norm["incluir"] == True].copy()
print(f"  Setores incluídos: {len(incl)}")
print(f"\n  {'Setor':<28} {'Abs (R$bi)':>12} {'Intens. (%VAB)':>15} {'Part. PIB':>10}")
print("  " + "-"*68)
for _, r in incl.sort_values("delta_vab_r$_bi").iterrows():
    print(f"  {r['setor']:<28} {r['delta_vab_r$_bi']:>12.3f} "
          f"{r['impacto_norm']:>15.3f}% {r['participacao_pib_pct']:>9.1f}%")

ordem_abs   = incl.sort_values("delta_vab_r$_bi")["setor"].tolist()
ordem_intens = incl.sort_values("impacto_norm")["setor"].tolist()
ordens_diferentes = ordem_abs != ordem_intens

print(f"\n  Ordenação absoluta ≠ ordenação intensidade: {'✓' if ordens_diferentes else '✗'}")
print(f"  (se iguais, o segundo painel não acrescenta informação)")

checks["V3_normalizacao_consistente"] = len(incl) == 7 and ordens_diferentes

In [ ]:
print("\n── V4: Cruzamento entre outputs ─────────────────────────")

df_base = pd.read_csv(PROC / "base_analitica_setorial_2023.csv")
pib_total = pd.read_csv(PROC / "pib_setorial_2012_2023.csv")
pib_rs    = pib_total[pib_total["ano"] == 2023]["pib_total"].values[0] * 1e6

delta_base_tabela = df_base[df_base["incluir"] == True]["delta_vab_r$_bi"].sum() * 1e9 / pib_rs * 100
delta_base_matriz = bl_0

diff = abs(delta_base_tabela - delta_base_matriz)
ok_cruzamento = diff < 0.001

print(f"  Cenário base na tabela:  {delta_base_tabela:.6f}%")
print(f"  Cenário base na matriz:  {delta_base_matriz:.6f}%")
print(f"  Diferença:               {diff:.6f} pp")
print(f"  Consistente: {'✓' if ok_cruzamento else '✗ DIVERGÊNCIA'}")

checks["V4_cruzamento_outputs"] = ok_cruzamento

In [ ]:
print("\n══ RESULTADO DA VALIDAÇÃO DE OUTPUTS ═══════════════════")
todos_ok = all(checks.values())

for k, v in checks.items():
    print(f"  {'✓' if v else '✗'} {k}")

print()
if todos_ok:
    print("  OUTPUTS VALIDADOS — valores dos gráficos consistentes com tabelas")
    print()
    print("  Valores confirmados para o carrossel (sem hardcode):")
    print(f"    Faixa diretamente afetada (41–44h): {pct_41_44h:.1f}% dos ocupados")
    print(f"    Faixa já em 40h exatas:             {pct_40h_exatas:.1f}% dos ocupados")
    print(f"    Cenário base ΔPIB:                  {delta_base_tabela:.3f}%")
    print(f"    Faixa baseline consistente:         [{df_mat[df_mat['categoria']=='baseline_consistente']['delta_pib_incl'].min():.3f}%, {bl_max:.3f}%]")
    print(f"    CNI distância do baseline:          {abs(bl_0 - (-0.700)):.3f} pp")
else:
    print("  OUTPUTS COM DIVERGÊNCIA — resolver antes de comunicar")